In [12]:
import torch
import torch.nn as nn
from torchvision import models

# RE-DÉFINITION DU DEVICE (au cas où le notebook l'a oublié)
device = (
    torch.device("cuda") if torch.cuda.is_available() else
    torch.device("mps") if torch.backends.mps.is_available() else
    torch.device("cpu")
)
print("Device utilisé :", device)

# 1. Chargement de ResNet18 avec ses poids par défaut (pré-entraînés)
weights = models.ResNet18_Weights.DEFAULT
model_resnet = models.resnet18(weights=weights)

# 2. On fige les paramètres du modèle (Freeze)
for param in model_resnet.parameters():
    param.requires_grad = False

# 3. On remplace la dernière couche linéaire pour l'adapter à CIFAR-10
num_ftrs = model_resnet.fc.in_features
model_resnet.fc = nn.Linear(num_ftrs, 10)

# 4. Envoi du modèle sur l'accélérateur (cuda/mps/cpu)
model_resnet = model_resnet.to(device)

# 5. Configuration de l'optimiseur (uniquement sur la couche fc)
optimizer = torch.optim.AdamW(model_resnet.fc.parameters(), lr=1e-3)
criterion = nn.CrossEntropyLoss()

print("ResNet18 prêt et configuré pour le Transfer Learning !")

Device utilisé : cuda
ResNet18 prêt et configuré pour le Transfer Learning !


In [13]:
# 1. Définition du Scheduler (divise le LR par 10 si la perte de test ne baisse pas pendant 2 époques)
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='min', patience=2, factor=0.1)

# 2. Définition de la classe EarlyStopping
class EarlyStopping:
    def __init__(self, patience=3, min_delta=0):
        self.patience = patience     # Nombre d'époques à attendre avant d'arrêter
        self.min_delta = min_delta   # Amélioration minimale requise
        self.best_loss = float('inf')
        self.counter = 0
        self.early_stop = False

    def __call__(self, val_loss):
        if val_loss < self.best_loss - self.min_delta:
            self.best_loss = val_loss
            self.counter = 0
            # Optionnel : sauvegarder les meilleurs poids ici si tu veux
        else:
            self.counter += 1
            print(f"--> EarlyStopping counter: {self.counter}/{self.patience}")
            if self.counter >= self.patience:
                self.early_stop = True

early_stopping = EarlyStopping(patience=3)

In [14]:
# Assure-toi d'avoir installé torchmetrics avant : !pip install torchmetrics
import torchmetrics
from tqdm import tqdm

# Initialisation des métriques
metric_collection = torchmetrics.MetricCollection([
    torchmetrics.Accuracy(task="multiclass", num_classes=10),
    torchmetrics.Precision(task="multiclass", num_classes=10, average="macro"),
    torchmetrics.F1Score(task="multiclass", num_classes=10, average="macro")
]).to(device)

def train_loop_resnet(dataloader, model, loss_fn, optimizer, epoch, writer):
    model.train()
    progress_bar = tqdm(dataloader, desc=f"Époque {epoch} [Train]", leave=False)
    for batch, (X, y) in enumerate(progress_bar):
        X, y = X.to(device), y.to(device)
        pred = model(X)
        loss = loss_fn(pred, y)
        
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        
        global_step = epoch * len(dataloader) + batch
        writer.add_scalar("Loss/Train_ResNet", loss.item(), global_step)

def test_loop_resnet(dataloader, model, loss_fn, epoch, writer):
    model.eval()
    metric_collection.reset() # Reset des métriques à chaque début de test
    total_loss = 0
    
    with torch.no_grad():
        for X, y in dataloader:
            X, y = X.to(device), y.to(device)
            pred = model(X)
            total_loss += loss_fn(pred, y).item()
            metric_collection.update(pred, y) # Envoi des prédictions à torchmetrics
            
    avg_loss = total_loss / len(dataloader)
    metrics = metric_collection.compute() # Calcul final
    
    # Logs TensorBoard
    writer.add_scalar("Loss/Test_ResNet", avg_loss, epoch)
    writer.add_scalar("Accuracy/Test_ResNet", metrics['MulticlassAccuracy'].item(), epoch)
    
    print(f"--> Époque {epoch} : Acc: {metrics['MulticlassAccuracy']:.4f} | F1: {metrics['MulticlassF1Score']:.4f} | Loss: {avg_loss:.4f}")
    return avg_loss

In [15]:
import torchvision
from torchvision import transforms
from torch.utils.tensorboard import SummaryWriter

# 1. On s'assure d'avoir TensorBoard prêt
writer = SummaryWriter("runs/resnet18_cifar10_experiment")

# 2. On recharge les transformations et les DataLoaders
cifar10_mean = (0.4914, 0.4822, 0.4465)
cifar10_std = (0.2023, 0.1994, 0.2010)

transform_train = transforms.Compose([
    transforms.RandomHorizontalFlip(),
    transforms.RandomCrop(32, padding=4),
    transforms.ToTensor(),
    transforms.Normalize(mean=cifar10_mean, std=cifar10_std)
])

transform_test = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize(mean=cifar10_mean, std=cifar10_std)
])

dataset_train = torchvision.datasets.CIFAR10(root='./data', train=True, transform=transform_train, download=True)
dataset_test = torchvision.datasets.CIFAR10(root='./data', train=False, transform=transform_test, download=True)

dataloader_train = torch.utils.data.DataLoader(dataset_train, batch_size=64, shuffle=True)
dataloader_test = torch.utils.data.DataLoader(dataset_test, batch_size=64, shuffle=False)

print("Dataloaders et TensorBoard initialisés avec succès !")

Dataloaders et TensorBoard initialisés avec succès !


In [16]:
def run_resnet_training():
    epochs = 10
    for t in range(epochs):
        print(f"\n=== Époque {t+1}/{epochs} ===")
        train_loop_resnet(dataloader_train, model_resnet, criterion, optimizer, t, writer)
        val_loss = test_loop_resnet(dataloader_test, model_resnet, criterion, t, writer)
        
        # Mise à jour du scheduler basé sur la perte de validation
        scheduler.step(val_loss)
        
        # Vérification du early stopping
        early_stopping(val_loss)
        if early_stopping.early_stop:
            print("Early stopping déclenché ! Fin de l'entraînement.")
            break
            
    writer.close()

run_resnet_training()


=== Époque 1/10 ===


--> Époque 0 : Acc: 0.3774 | F1: 0.3766 | Loss: 1.8185

=== Époque 2/10 ===


--> Époque 1 : Acc: 0.3651 | F1: 0.3543 | Loss: 1.8348
--> EarlyStopping counter: 1/3

=== Époque 3/10 ===


--> Époque 2 : Acc: 0.3914 | F1: 0.3821 | Loss: 1.7589

=== Époque 4/10 ===


--> Époque 3 : Acc: 0.3957 | F1: 0.3788 | Loss: 1.7674
--> EarlyStopping counter: 1/3

=== Époque 5/10 ===


--> Époque 4 : Acc: 0.3955 | F1: 0.3913 | Loss: 1.7737
--> EarlyStopping counter: 2/3

=== Époque 6/10 ===


--> Époque 5 : Acc: 0.4037 | F1: 0.4021 | Loss: 1.7397

=== Époque 7/10 ===


--> Époque 6 : Acc: 0.4121 | F1: 0.4050 | Loss: 1.7418
--> EarlyStopping counter: 1/3

=== Époque 8/10 ===


--> Époque 7 : Acc: 0.3913 | F1: 0.3911 | Loss: 1.7749
--> EarlyStopping counter: 2/3

=== Époque 9/10 ===


--> Époque 8 : Acc: 0.3919 | F1: 0.3809 | Loss: 1.7733
--> EarlyStopping counter: 3/3
Early stopping déclenché ! Fin de l'entraînement.


In [18]:
model_resnet.eval()
# Création d'un tenseur de test simulant une image d'entrée CIFAR-10
dummy_input = torch.randn(1, 3, 32, 32, device=device)

onnx_path = "resnet18_cifar10.onnx"
torch.onnx.export(
    model_resnet, 
    dummy_input, 
    onnx_path, 
    export_params=True, 
    opset_version=11, 
    do_constant_folding=True,
    input_names=['input'], 
    output_names=['output'],
    dynamic_axes={'input': {0: 'batch_size'}, 'output': {0: 'batch_size'}}
)
print(f"Modèle exporté avec succès sous : {onnx_path}")

/tmp/ipykernel_35462/2190818252.py:6: UserWarning: # 'dynamic_axes' is not recommended when dynamo=True, and may lead to 'torch._dynamo.exc.UserError: Constraints violated.' Supply the 'dynamic_shapes' argument instead if export is unsuccessful.
  torch.onnx.export(
W0610 09:52:14.317000 35462 torch/onnx/_internal/exporter/_compat.py:133] Setting ONNX exporter to use operator set version 18 because the requested opset_version 11 is a lower version than we have implementations for. Automatic version conversion will be performed, which may not be successful at converting to the requested version. If version conversion is unsuccessful, the opset version of the exported model will be kept at 18. Please consider setting opset_version >=18 to leverage latest ONNX features


[torch.onnx] Obtain model graph for `ResNet([...]` with `torch.export.export(..., strict=False)`...
[torch.onnx] Obtain model graph for `ResNet([...]` with `torch.export.export(..., strict=False)`... ✅
[torch.onnx] Run decompositions...


/usr/lib/python3.12/copyreg.py:99: FutureWarning: `isinstance(treespec, LeafSpec)` is deprecated, use `isinstance(treespec, TreeSpec) and treespec.is_leaf()` instead.
  return cls.__new__(cls, *args)
The model version conversion is not supported by the onnxscript version converter and fallback is enabled. The model will be converted using the onnx C API (target version: 11).
Failed to convert the model to the target version 11 using the ONNX C API. The model was not modified
Traceback (most recent call last):
  File "/home/voilacter/Documents/devops/ia/terrain/.venv/lib/python3.12/site-packages/onnxscript/version_converter/__init__.py", line 137, in call
    converted_proto = _c_api_utils.call_onnx_api(
                      ^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/home/voilacter/Documents/devops/ia/terrain/.venv/lib/python3.12/site-packages/onnxscript/version_converter/_c_api_utils.py", line 65, in call_onnx_api
    result = func(proto)
             ^^^^^^^^^^^
  File "/home/voilacter/Doc

[torch.onnx] Run decompositions... ✅
[torch.onnx] Translate the graph into ONNX...
[torch.onnx] Translate the graph into ONNX... ✅
[torch.onnx] Optimize the ONNX graph...
[torch.onnx] Optimize the ONNX graph... ✅
Modèle exporté avec succès sous : resnet18_cifar10.onnx
